<a href="https://colab.research.google.com/github/thecodernisha/fake-review-detection-using-BERT-and-EfficientNet/blob/main/final_research_file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Still CPU'}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Device Detection:")
print(f"   GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(f"   Using: CPU (Slow - GPU recommended for faster training)")
print(f"   Active Device: {device}")

In [ ]:
!pip install torch torchvision transformers datasets Pillow requests scikit-learn matplotlib seaborn -q

print("All libraries installed successfully")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import requests
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torchvision import models, transforms
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports successful")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install torch torchvision transformers datasets Pillow requests scikit-learn matplotlib seaborn kagglehub -q

print("All libraries installed successfully")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import requests
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torchvision import models, transforms
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import os
from io import BytesIO
import pickle
import kagglehub
import shutil

print("All imports successful")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

print("Loading AiGen-FoodReview from Hugging Face...")
dataset = load_dataset("davanstrien/AiGen-FoodReview")

print(dataset)
print("Features:", dataset["train"].features)

train_data = dataset["train"].to_pandas()
val_data = dataset["valid"].to_pandas()
test_data = dataset["test"].to_pandas()


print(f"Original train size: {len(train_data)}")
print(f"Original validation size: {len(val_data)}")
print(f"Original test size: {len(test_data)}")
print("Label distribution in training:")
print(train_data['label'].value_counts())

"""
min_count = min(train_data['label'].value_counts())
sample_size = min(min_count, 1500)
df_real = train_data[train_data['label'] == 0].sample(n=sample_size, random_state=42)
df_fake = train_data[train_data['label'] == 1].sample(n=sample_size, random_state=42)
train_data = pd.concat([df_real, df_fake]).sample(frac=1, random_state=42).reset_index(drop=True)
"""

train_df = train_data
val_df = val_data

In [ ]:
class GatedMultiModalFakeReviewDetector(nn.Module):
    def __init__(self, freeze_bert=True, freeze_effnet=True, dropout_prob=0.5):
        super().__init__()
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        self.effnet = models.efficientnet_b0(weights='DEFAULT')
        self.effnet.classifier = nn.Identity()

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
        if freeze_effnet:
            for param in self.effnet.parameters():
                param.requires_grad = False

        self.text_proj = nn.Linear(768, 256)
        self.image_proj = nn.Linear(1280, 256)

        self.gate = nn.Sequential(
            nn.Linear(256 + 256, 2),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 + 256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(128, 2)
        )

    def forward(self, ids, mask, image, image_valid):
        bert_out = self.bert(ids, attention_mask=mask)
        text_feat = self.text_proj(bert_out.pooler_output)

        image_feat = self.effnet(image)
        image_feat = image_feat.view(image_feat.size(0), -1)
        image_feat = self.image_proj(image_feat)

        image_feat = image_feat * image_valid

        combined = torch.cat([text_feat, image_feat], dim=1)
        gates = self.gate(combined)
        gated_text = text_feat * gates[:, 0:1]
        gated_image = image_feat * gates[:, 1:2]
        fused = torch.cat([gated_text, gated_image], dim=1)

        return self.classifier(fused)

In [ ]:
class MultiModalFakeReviewDataset(Dataset):
    def __init__(self, dataframe, tokenizer, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        text = str(row['text'])

        encoded = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        image = row['image']

        if image is None or not isinstance(image, Image.Image):
            img_tensor = torch.zeros(3, 224, 224)
            image_valid = torch.zeros(1)
        else:
            if self.transform:
                img_tensor = self.transform(image)
            else:
                img_tensor = transforms.ToTensor()(image)
            image_valid = torch.ones(1)

        return {
            'ids': encoded['input_ids'].flatten(),
            'mask': encoded['attention_mask'].flatten(),
            'image': img_tensor,
            'image_valid': image_valid,
            'label': torch.tensor(row['label'], dtype=torch.long)
        }

In [ ]:
img_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_dataset = MultiModalFakeReviewDataset(train_df, tokenizer, img_transforms)
val_dataset = MultiModalFakeReviewDataset(val_df, tokenizer, img_transforms)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = GatedMultiModalFakeReviewDetector().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)

print("Model ready.")
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/fake_review_models'
os.makedirs(MODEL_PATH, exist_ok=True)
BEST_MODEL_PATH = os.path.join(MODEL_PATH, 'best_multimodal_model.pth')

EPOCHS = 15
PATIENCE = 3
best_val_acc = 0.0
patience_counter = 0

print(f"Starting training for up to {EPOCHS} epochs...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        ids = batch['ids'].to(device)
        mask = batch['mask'].to(device)
        images = batch['image'].to(device)
        img_valid = batch['image_valid'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(ids, mask, images, img_valid)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['ids'].to(device)
            mask = batch['mask'].to(device)
            images = batch['image'].to(device)
            img_valid = batch['image_valid'].to(device)
            labels = batch['label'].to(device)

            outputs = model(ids, mask, images, img_valid)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  --> New best model saved (acc={val_acc:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

print("Training complete.")

In [ ]:
import os
import torch

MODEL_PATH = '/content/drive/MyDrive/fake_review_models'
BEST_MODEL_PATH = os.path.join(MODEL_PATH, 'best_multimodal_model.pth')

model = GatedMultiModalFakeReviewDetector().to(device)

model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()
print(f"Best model loaded from {BEST_MODEL_PATH}")

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        ids = batch['ids'].to(device)
        mask = batch['mask'].to(device)
        images = batch['image'].to(device)
        img_valid = batch['image_valid'].to(device)
        labels = batch['label'].to(device)

        outputs = model(ids, mask, images, img_valid)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print("\n" + "="*60)
print("         MULTIMODAL MODEL PERFORMANCE REPORT")
print("="*60)
print(f"   Accuracy : {acc:.4f} ({acc*100:.1f}%)")
print(f"   Precision: {prec:.4f} ({prec*100:.1f}%)")
print(f"   Recall   : {rec:.4f} ({rec*100:.1f}%)")
print(f"   F1-Score : {f1:.4f} ({f1*100:.1f}%)")
print("="*60)

print("\nDetailed Classification Report:")
print(classification_report(all_labels, all_preds, target_names=['REAL', 'FAKE']))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['REAL','FAKE'], yticklabels=['REAL','FAKE'])
plt.title('Confusion Matrix - Gated Multimodal Model')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_PATH, 'confusion_matrix_multimodal.png'), dpi=150)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

model.eval()
all_probs = []
all_labels = []
with torch.no_grad():
    for batch in val_loader:
        ids = batch['ids'].to(device)
        mask = batch['mask'].to(device)
        images = batch['image'].to(device)
        img_valid = batch['image_valid'].to(device)
        labels = batch['label'].to(device)
        outputs = model(ids, mask, images, img_valid)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

auc = roc_auc_score(all_labels, all_probs)
fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend()
plt.show()

In [ ]:
test_dataset = MultiModalFakeReviewDataset(test_data, tokenizer, img_transforms)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model.eval()
print("Testing on random samples from the test set:\n")
for i in range(5):
    batch = test_dataset[i]
    ids = batch['ids'].unsqueeze(0).to(device)
    mask = batch['mask'].unsqueeze(0).to(device)
    image = batch['image'].unsqueeze(0).to(device)
    img_valid = batch['image_valid'].unsqueeze(0).to(device)
    true_label = batch['label'].item()

    with torch.no_grad():
        output = model(ids, mask, image, img_valid)
        pred = torch.argmax(output, dim=1).item()
        confidence = torch.softmax(output, dim=1)[0][pred].item()

    review_text = tokenizer.decode(ids.squeeze(), skip_special_tokens=True)
    print(f"Review: {review_text[:100]}...")
    print(f"True label: {'FAKE' if true_label==1 else 'REAL'}")
    print(f"Prediction: {'FAKE' if pred==1 else 'REAL'} (confidence: {confidence:.1%})")
    print("-"*60)

In [ ]:
test_dataset = MultiModalFakeReviewDataset(test_data, tokenizer, img_transforms)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

model.eval()
all_test_preds = []
all_test_labels = []

with torch.no_grad():
    for batch in test_loader:
        ids = batch['ids'].to(device)
        mask = batch['mask'].to(device)
        images = batch['image'].to(device)
        img_valid = batch['image_valid'].to(device)
        labels = batch['label'].to(device)

        outputs = model(ids, mask, images, img_valid)
        preds = torch.argmax(outputs, dim=1)
        all_test_preds.extend(preds.cpu().numpy())
        all_test_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(all_test_labels, all_test_preds)
print(f"\n=== Test Set Results ===")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")
print("\nDetailed Test Classification Report:")
print(classification_report(all_test_labels, all_test_preds, target_names=['REAL', 'FAKE']))

cm_test = confusion_matrix(all_test_labels, all_test_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=['REAL','FAKE'], yticklabels=['REAL','FAKE'])
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_PATH, 'confusion_matrix_test.png'), dpi=150)
plt.show()

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'model_class': 'GatedMultiModalFakeReviewDetector',
    'val_accuracy': best_val_acc,
    'test_accuracy': test_acc
}, os.path.join(MODEL_PATH, 'complete_multimodal_model.pth'))

print(f"Complete model saved at {MODEL_PATH}/complete_multimodal_model.pth")